# ClipKontrast on Kaggle or Google Colab

This notebook runs the complete ClipKontrast app in the notebook runtime. All generated data, downloaded models, source media, and clips stay under `PROJECT/data`. For the fastest link workflow, the app uses YouTube's timed transcript first and downloads only the selected clip sections when an AI key is configured.

Use a GPU runtime when available. The GPU is useful for local speech-to-text fallback; YouTube transcript-first jobs can begin without downloading the full source audio.

In [ ]:
# Set PROJECT to the folder containing this repository.
# Examples: /content/ClipKontrast (Colab), /kaggle/working/ClipKontrast (Kaggle).
from pathlib import Path
import os
PROJECT = Path(os.environ.get('CLIPKONTRAST_PROJECT', '/content/ClipKontrast')).resolve()
if not (PROJECT / 'studio').is_dir():
    raise FileNotFoundError(f'Put the ClipKontrast folder at {PROJECT}, or set CLIPKONTRAST_PROJECT first.')
os.chdir(PROJECT)
DATA = PROJECT / 'data'
DATA.mkdir(exist_ok=True)
print('Project:', PROJECT)
print('Data:', DATA)

In [ ]:
# System and Python dependencies. Re-run safely after a runtime restart.
import subprocess, sys
subprocess.run(['bash', '-lc', 'apt-get -qq update && apt-get -qq install -y ffmpeg'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print('Dependencies ready')

In [ ]:
# Notebook-safe runtime settings. Change the AI provider/model/key below if desired.
import getpass
os.environ.update({
    'CLIPKONTRAST_DATA': str(DATA),
    'CLIPKONTRAST_HOST': '0.0.0.0',
    'CLIPKONTRAST_PORT': '8765',
    'CLIPKONTRAST_ACCOUNTS': '0',
    'CLIPKONTRAST_UPDATE_CHECK': '0',
    'CLIPKONTRAST_FAST_URLS': '1',
    'CLIPKONTRAST_YT_TRANSCRIPT': '1',
    'CLIPKONTRAST_FETCH_MODEL': '0',
    'PYTHONUNBUFFERED': '1',
})
# Uncomment and fill these when using an AI provider.
# os.environ['CLIPKONTRAST_AI_PROVIDER'] = 'gemini'
# os.environ['CLIPKONTRAST_AI_MODELS'] = 'gemini-2.5-flash'
# os.environ['CLIPKONTRAST_AI_KEY'] = getpass.getpass('AI key (hidden): ')
print('Runtime configured; data remains inside', DATA)

In [ ]:
# Optional: set this True to download the speech model before opening the app.
# Leave it False for the fastest startup; transcript-first YouTube jobs do not need it.
PRELOAD_ASR_MODEL = False
if PRELOAD_ASR_MODEL:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'studio.fetch_model'], check=True)
    print('Speech model is ready')
else:
    print('Speech model preload skipped')

In [ ]:
# Start the complete web app. Keep this cell running.
import subprocess, sys, time, urllib.request
server = subprocess.Popen([sys.executable, '-m', 'studio.server'], cwd=PROJECT, env=os.environ.copy())
for _ in range(60):
    try:
        urllib.request.urlopen('http://127.0.0.1:8765/healthz', timeout=2)
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError('ClipKontrast did not start; inspect the server output above.')
print('ClipKontrast is running on port 8765')

## Open the app

In Google Colab, run the next cell to open port 8765 in a notebook tab. In Kaggle, use the notebook's **Open session / port 8765** control when available, or create a temporary tunnel according to your Kaggle environment policy.

In [ ]:
try:
    from google.colab import output
    output.serve_kernel_port_as_window(8765)
except ImportError:
    print('Colab helper not available. Open the notebook runtime port 8765 from the Kaggle UI.')

In [ ]:
# Stop the app cleanly when finished.
server.terminate()
server.wait(timeout=15)
print('ClipKontrast stopped; files remain in', DATA)